# Lab 9 — Anscombe's quartet과 상관계수의 한계

**확률통계 · Topic 9 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 요약 통계량이 같은데 **모양이 전혀 다른** 데이터를 직접 확인한다.
2. **상관계수가 0인데 완벽히 종속인** 데이터를 직접 만들어본다.
3. 결합분포에서 **주변분포·조건부분포**를 계산한다.

⏱ **예상 소요 시간: 35분**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. Anscombe's quartet (1973)

네 데이터 세트다. 먼저 **숫자만** 보자.

In [ ]:
x_common = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)

anscombe = {
    "I":   (x_common, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    "II":  (x_common, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    "III": (x_common, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    "IV":  (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], dtype=float),
            np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}

print(f"{'set':<5}{'mean x':>9}{'mean y':>9}{'var x':>9}{'var y':>9}")
for name, (x, y) in anscombe.items():
    print(f"{name:<5}{x.mean():9.2f}{y.mean():9.2f}{x.var(ddof=1):9.2f}{y.var(ddof=1):9.2f}")

### 실습 1 — 상관계수와 회귀직선도 같은지 확인

`np.corrcoef(x, y)` 는 2x2 **상관행렬**을 돌려준다. 우리가 원하는 값은 `[0, 1]` 위치다.
`np.polyfit(x, y, 1)` 은 직선을 맞춰 `[기울기, 절편]` 을 돌려준다.

In [ ]:
print(f"{'set':<5}{'corr':>9}{'slope':>9}{'intercept':>11}")
for name, (x, y) in anscombe.items():
    # TODO 1: 상관계수와 회귀직선의 기울기·절편을 구하세요
    #         힌트: np.corrcoef(x, y)[0, 1] / np.polyfit(x, y, 1)
    r = 0.0
    slope, intercept = 0.0, 0.0
    print(f"{name:<5}{r:9.3f}{slope:9.3f}{intercept:11.3f}")

> **네 줄이 전부 똑같이 나왔는가?** 그렇다면 이 네 데이터는 숫자로는 구별할 수 없다.

### 실습 2 — 이제 그림을 그리자

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), sharex=True, sharey=True)
for ax, (name, (x, y)) in zip(axes, anscombe.items()):
    # TODO 2: 산점도와 회귀직선을 그리세요
    #         힌트: ax.plot(x, y, "o") 그리고 np.polyfit 결과로 직선

    ax.set_title(name)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

😲 **I은 평범한 직선 관계, II는 곡선, III는 이상치 하나, IV는 x가 거의 한 값.**
숫자는 같은데 이야기가 전혀 다르다.

> **분석을 시작할 때 그림부터 그리는 습관**이 여기서 나온다.

## Part 2. 상관 0인데 완벽히 종속인 데이터 만들기

### 실습 3

In [ ]:
x = rng.uniform(-1, 1, 2000)
# TODO 3: y를 x의 제곱으로 바꾸세요 (지금은 y = x 라서 상관계수가 1이 나온다)
y = x.copy()

r = np.corrcoef(x, y)[0, 1]

plt.figure(figsize=(5, 4))
plt.plot(x, y, ".", ms=4)
plt.xlabel("x")
plt.ylabel("y = x^2")
plt.title(f"corr = {r:.4f}")
plt.show()

print(f"상관계수: {r:.4f}")

> **상관계수는 직선 관계만 잰다.** 포물선의 왼쪽(감소)과 오른쪽(증가)이 상쇄되어 0이 된다.

### 실습 4 — 다른 반례들도 만들어보기

In [ ]:
cases = {}

t = rng.uniform(0, 2 * np.pi, 2000)
cases["circle"] = (np.cos(t), np.sin(t))

x = rng.uniform(-1, 1, 2000)
cases["abs"] = (x, np.abs(x))

x = rng.normal(size=2000)
cases["variance only"] = (x, rng.normal(0, 1, 2000) * np.abs(x))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (name, (xx, yy)) in zip(axes, cases.items()):
    ax.plot(xx, yy, ".", ms=3, color="darkorange")
    # TODO 4: 제목에 상관계수를 표시하세요
    #         힌트: np.corrcoef(xx, yy)[0, 1]
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Part 3. 결합분포에서 주변·조건부 구하기

동전 두 개. $X$ = 첫 번째가 앞면인가(0/1), $Y$ = 앞면의 총 개수.

| $p_{X,Y}$ | $Y=0$ | $Y=1$ | $Y=2$ |
|---|---|---|---|
| $X=0$ | 1/4 | 1/4 | 0 |
| $X=1$ | 0 | 1/4 | 1/4 |

### 실습 5 — 표에서 주변분포 뽑아내기

In [ ]:
joint = np.array([[0.25, 0.25, 0.00],
                  [0.00, 0.25, 0.25]])

# TODO 5: 행/열 방향으로 더해 주변분포를 구하세요
#         힌트: joint.sum(axis=1) 과 joint.sum(axis=0)
p_x = np.zeros(2)
p_y = np.zeros(3)

print("P_X =", p_x, " 합:", p_x.sum())
print("P_Y =", p_y, " 합:", p_y.sum())

outer = np.outer(p_x, p_y)
print("독립인가?", np.allclose(joint, outer))

**독립이 아니다.** $p_{X,Y}(0,2) = 0$ 인데 $p_X(0)p_Y(2) = 0.125$ 이기 때문이다.
첫 동전 결과가 총 개수에 대한 정보를 준다.

---

## 마무리 — 자가 점검

- [ ] 요약 통계량이 같아도 데이터가 전혀 다를 수 있음을 확인했다
- [ ] 상관계수가 0인데 종속인 예를 직접 만들었다
- [ ] 결합분포에서 주변분포와 조건부분포를 계산할 수 있다
- [ ] 독립 여부를 "결합 = 주변의 곱"으로 판정할 수 있다

**오늘 배운 것을 한 문장으로.**

> (여기에 작성)

### 📌 HW 4 — `hw/T09_hw.md` · 기한은 공지 확인